# TieredKV: Sampreeth's full / H2O target-model matrix

Run full cache and H2O at **10%, 15%, 20%, 25%** retained-KV budgets on the same pinned model and fixed GSM8K subset. **No dictionary file is required.** Select a GPU runtime with at least 10 GiB VRAM for the default 3B model.

Five target-model prompts first check stock/full equality and shared H2O versus the independent reference at every budget. This notebook is an experiment runner, not a completed benchmark. Outputs and failure logs are saved to Drive.


In [ ]:
from pathlib import Path
import os, subprocess, sys

BRANCH = "b-allocator-phase4-final"
PROJECT = Path(os.environ.get("TIEREDKV_PROJECT","/content/KV-cache-h2o")).resolve()
if not PROJECT.exists():
    subprocess.run(["git","clone","--branch",BRANCH,
                    "https://github.com/samarth2006-28/KV-cache.git",str(PROJECT)],check=True)
else:
    current = subprocess.check_output(["git","branch","--show-current"],cwd=PROJECT,text=True).strip()
    dirty = subprocess.check_output(["git","status","--porcelain"],cwd=PROJECT,text=True).strip()
    if current != BRANCH or dirty:
        raise RuntimeError("Existing checkout is another branch or has edits. Choose a new PROJECT path.")
    subprocess.run(["git","pull","--ff-only"],cwd=PROJECT,check=True)
os.chdir(PROJECT)
# Notebook kernels can retain an absolute sys.path after changing folders.
if str(PROJECT) not in sys.path:
    sys.path.insert(0,str(PROJECT))
import importlib
importlib.invalidate_caches()
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"],check=True)
print("Code commit:",subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip())

In [ ]:
import torch
from importlib.metadata import version
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory/2**30,1), "GiB")
print("Torch",torch.__version__,"Transformers",version("transformers"))
if version("transformers") != "4.49.0":
    raise RuntimeError("Setup did not install Transformers 4.49.0. Re-run the setup cell.")
if props.total_memory < 10*2**30:
    raise RuntimeError("This 3B workflow requires at least 10 GiB VRAM for weights and inference buffers.")

## Model and settings

Choose the same MODEL_ID, exact MODEL_REVISION, N, SEED and MAX_NEW_TOKENS as the team's other experiments before looking at answers. If MODEL_REVISION is blank, one exact current commit is resolved and recorded. To share a cohort with Phase 4, copy that notebook's recorded REVISION here.

For a gated model, authenticate in your runtime with huggingface_hub.notebook_login(). Do not put access tokens in notebook source.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"
MODEL_REVISION = ""  # use the team's agreed exact model SHA
N = 20
SEED = 0
MAX_NEW_TOKENS = 256
SAVE_ROOT = Path("/content/drive/MyDrive/tieredkv_h2o_runs")


In [ ]:
import json
from huggingface_hub import HfApi, snapshot_download
REVISION = MODEL_REVISION or HfApi().model_info(MODEL_ID).sha
import re
if not re.fullmatch(r"[0-9a-f]{40}",REVISION):
    raise ValueError("Use an exact model commit SHA.")
print("Model:",MODEL_ID,"commit:",REVISION)
MODEL = snapshot_download(repo_id=MODEL_ID,revision=REVISION,
    ignore_patterns=["*.bin","*.gguf","*.h5","*.ot","original/*"])
print("Pinned snapshot ready:",MODEL)


## Optional offline regression checks

These test code compatibility. Five actual target-model generation gates run separately before accuracy measurement. Local tiny-model tests are not target-model results.


In [ ]:
os.environ.pop("TIEREDKV_MODEL",None)
subprocess.run([sys.executable,"-m","pytest","-q","tests/test_allocator.py",
                "tests/test_h2o_reference.py","tests/test_h2o_experiment.py"],check=True,cwd=str(PROJECT))


## Run the four budgets and full baseline

The same model stays loaded across cells in this matrix. First, a twenty-prompt target-model prefill probe saves raw sums, valid causal-query counts and normalized scores under attention/. Raw attention mass and normalized score share are reported separately. Raw predictions, manifests, the append-only CSV, per-problem audits, Wilson intervals, four paired full-vs-H2O tests and plots are saved to a unique Drive directory. All five cells must pass the budget audit before h2o_completed becomes true. The allowed target difference is one percentage point, following the team plan.

H2O uses raw accumulated attention, not mean-normalized TieredKV scores. This avoids the compressed encoder/decoder bottleneck. Full prefill and eager attention still need temporary memory; retained KV is not peak VRAM. Timing numbers are measurements, not guaranteed speedups.

If a process fails, the next cells still let you inspect and download the partial bundle. The .log includes the original traceback. Start a new output directory on retry; no previous results are overwritten.


In [ ]:
from datetime import datetime, timezone
import uuid
from tieredkv.notebook_support import run_logged
SAVE_ROOT.mkdir(parents=True,exist_ok=True)
OUT = SAVE_ROOT/("h2o_"+datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")+"_"+uuid.uuid4().hex[:8])
command = [sys.executable,"-u","-m","tieredkv.h2o_experiment",
           "--model",MODEL,"--model-id",MODEL_ID,"--model-revision",REVISION,
           "--out",str(OUT),"--n",str(N),"--seed",str(SEED),
           "--max-new-tokens",str(MAX_NEW_TOKENS)]
LOG = Path(str(OUT)+".log")
print("Starting H2O matrix. Live output below; full log:",LOG,flush=True)
run_logged(command,LOG,result_dir=OUT,cwd=str(PROJECT))
print("Results:",OUT)
print("Archive:",str(OUT)+".zip")


In [ ]:
report_path = OUT/"experiment.json"
if not report_path.exists():
    raise RuntimeError("Preflight failed before result creation; inspect the preceding error.")
report = json.loads(report_path.read_text())
print("Status:",report["status"])
print("H2O matrix completed:",report["h2o_completed"])
for method,cell in report.get("cells",{}).items():
    print(method, f'{cell["strict_correct"]}/{cell["n"]} strict correct',
          f'mean KV={100*cell["mean_budget"]:.3f}%',
          "eligible=",cell["eligible"],"budget failures=",cell["budget_failures"])
for pair in report.get("paired_mcnemar",[]):
    print(pair["a"],"vs",pair["b"],"adjusted p=",pair["p_bonferroni"])
if report.get("error"):
    print("Error:",report["error"])
if (OUT/"table.md").exists():
    from IPython.display import display, Markdown
    display(Markdown((OUT/"table.md").read_text()))
print("Completion requires all five cells and their actual budget audits. No automatic superiority claim.")


In [ ]:
from google.colab import files
archive = Path(str(OUT)+".zip")
if archive.is_file():
    files.download(str(archive))
else:
    print("No archive yet. The experiment directory and progress files remain on Drive.")